# Unity Catalog SQL functions — cogsail LimsResult

Import this notebook to validate `generator.create_sql_function` on the cogsail `LimsResult` view.

Every case follows the same path:

1. Write an analyst query against the registered view. The literals in that query are only examples.
2. `create_sql_function` checks that the query can be pushed, turns the literals into arguments, and runs `CREATE OR REPLACE FUNCTION` on the SQL warehouse.
3. Call that function with other argument values. Each call is timed. A different value must return a different count, which shows the argument reached CDF.

Upload both `0.0.0` wheels to `/Workspace/Users/fredrik.holm@cognitedata.com/wheels/` first. `%pip` and `%restart_python` are their own cells.

Seed counts used below: lab A 60,000, lab B 40,000. Each `componentName` is 6,000 in lab A and 4,000 in lab B.


## 1. Install the local wheels


In [ ]:
%pip install --force-reinstall /Workspace/Users/fredrik.holm@cognitedata.com/wheels/cognite_pygen_spark-0.0.0-py3-none-any.whl /Workspace/Users/fredrik.holm@cognitedata.com/wheels/cognite_databricks-0.0.0-py3-none-any.whl


In [ ]:
%restart_python


In [ ]:
import cognite.databricks as db
import cognite.pygen_spark as ps

print(ps.__version__, db.__version__)
assert ps.__version__ == "0.0.0"
assert db.__version__ == "0.0.0"


## 2. Cogsail client

A bearer token is enough to read CDF. Writing the secret scope still needs client credentials, or an existing scope from an earlier registration.


In [ ]:
import tomllib
from pathlib import Path

from cognite.client import CogniteClient
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, Token
from databricks.sdk import WorkspaceClient

TOML_PATH = "/Workspace/Users/fredrik.holm@cognitedata.com/toml/cog-sail.toml"

CATALOG = "f0connectortest"
SECRET_SCOPE = "cdf_dm_dom_lims_result_limsresult_dom"
DM_SPACE = "dm_dom_lims_result"
MODEL_EXTERNAL_ID = "LimsResult_DOM"
MODEL_VERSION = "v1"
SCHEMA = f"{DM_SPACE}_{MODEL_EXTERNAL_ID.lower()}_{MODEL_VERSION}"
VIEW = f"{CATALOG}.{SCHEMA}.LimsResult"
LAB_A = "inst_lims_result_lab_a"
LAB_B = "inst_lims_result_lab_b"
OUTPUT_DIR = "/local_disk0/tmp/pygen_udtf"

with Path(TOML_PATH).open("rb") as handle:
    data = tomllib.load(handle)
cognite = data["cognite"]
runtime = data.get("fn_btp_hierarchy_runtime") or {}
triggers = data.get("hierarchy_workflow_triggers") or {}
secrets = data.get("hierarchy_workflow_trigger_secrets") or {}
client_id = runtime.get("client_id") or cognite.get("client_id") or triggers.get("ingest_trigger_client_id")
client_secret = (
    runtime.get("client_secret") or cognite.get("client_secret") or secrets.get("ingest_trigger_client_secret")
)
tenant_id = cognite.get("tenant_id") or cognite.get("idp_tenant_id")
base_url = f"https://{cognite['cdf_cluster']}.cognitedata.com"

if client_id and client_secret and tenant_id:
    credentials = OAuthClientCredentials(
        token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        client_id=client_id,
        client_secret=client_secret,
        scopes=[f"{base_url}/.default"],
    )
else:
    bearer_token = cognite.get("bearer_token")
    if not isinstance(bearer_token, str) or not bearer_token:
        raise ValueError("TOML needs client credentials or cognite.bearer_token")
    credentials = Token(lambda: bearer_token)

client = CogniteClient(
    ClientConfig(
        client_name="sql-function-latency",
        project=cognite["project"],
        credentials=credentials,
        base_url=base_url,
    )
)
workspace_client = WorkspaceClient()
print("project:", client.config.project)
print("view:", VIEW)


## 3. Register the view from this wheel


In [ ]:
from cognite.client.data_classes.data_modeling.ids import DataModelId
from cognite.databricks import generate_udtf_notebook

generator = generate_udtf_notebook(
    DataModelId(space=DM_SPACE, external_id=MODEL_EXTERNAL_ID, version=MODEL_VERSION),
    client,
    workspace_client=workspace_client,
    catalog=CATALOG,
    schema=SCHEMA,
    output_dir=OUTPUT_DIR,
)
if client_id and client_secret and tenant_id:
    generator.secret_helper.set_cdf_credentials(
        scope_name=SECRET_SCOPE,
        project=client.config.project,
        cdf_cluster=cognite["cdf_cluster"],
        client_id=client_id,
        client_secret=client_secret,
        tenant_id=tenant_id,
        base_url=client.config.base_url,
    )
else:
    print("Bearer token only. Reusing the secret scope from the last registration.")

secret_keys = sorted(item.key for item in workspace_client.secrets.list_secrets(scope=SECRET_SCOPE))
print("secret keys:", secret_keys)
assert "base_url" in secret_keys
udtfs = generator.register_udtfs(secret_scope=SECRET_SCOPE, if_exists="replace")
views = generator.register_views(secret_scope=SECRET_SCOPE, if_exists="replace")
print("registered UDTFs:", udtfs.total_count)
print("registered views:", [item.view_name for item in views.registered_views])


## 4. Create a function, then call it

`exercise` is the whole user flow. It creates the function, reads the signature back from Unity Catalog, then runs each call with `spark.sql` on this notebook session and records the latency.


In [ ]:
import time

from cognite.databricks.data_model_query_rewriter import QueryNotPushdownCompatible
from databricks.sdk.errors import NotFound

import sys

results: list[tuple[str, bool, str]] = []
timings: list[tuple[str, float | None, int, bool]] = []


def check(name: str, ok: bool, detail: str = "") -> bool:
    results.append((name, ok, detail))
    mark = "PASS" if ok else "FAIL"
    suffix = f"  — {detail}" if detail else ""
    print(f"{mark}  {name}{suffix}")
    sys.stdout.flush()
    return ok


def run_function(statement: str) -> list[dict[str, object]]:
    rows = spark.sql(statement).collect()
    return [row.asDict() for row in rows]


def one_count(expected: int):
    def expect(rows: list[dict[str, str | None]]) -> bool:
        return len(rows) == 1 and str(rows[0].get("count_externalId")) == str(expected)

    return expect


def grouped(expected_rows: int, expected_count: str):
    def expect(rows: list[dict[str, str | None]]) -> bool:
        return len(rows) == expected_rows and {str(row.get("count_externalId")) for row in rows} == {expected_count}

    return expect


def exercise(title: str, function_name: str, analyst_sql: str, calls: list[tuple[str, str, object]], absent: list[str]) -> str:
    print(f"\n======== {title}")
    print("analyst SQL:")
    print(analyst_sql.strip())
    created = generator.create_sql_function(function_name, analyst_sql, secret_scope=SECRET_SCOPE)
    signature = ", ".join(f"{arg.name} {arg.sql_type}" for arg in created.arguments)
    print(f"created {created.full_name}({signature})")
    info = workspace_client.functions.get(created.full_name)
    listed = [parameter.name for parameter in (info.input_params.parameters if info.input_params else [])]
    check(f"{function_name} is in Unity Catalog", listed == [arg.name for arg in created.arguments], signature)
    check(
        f"{function_name} keeps secrets in the body",
        "SECRET(" in created.statement and "SQL SECURITY DEFINER" in created.statement,
    )
    leaked = [value for value in absent if value in created.statement]
    check(f"{function_name} arguments are not baked in", not leaked, ", ".join(leaked))
    for label, statement, expect in calls:
        print(f"\n--- {label}")
        print(statement.strip())
        sys.stdout.flush()
        try:
            started = time.perf_counter()
            rows = run_function(statement.replace("{fn}", created.full_name))
            elapsed = time.perf_counter() - started
            for row in rows[:12]:
                shown = {key: row.get(key) for key in ("space", "external_id", "componentName", "count_externalId", "min_numericValue", "max_numericValue") if key in row}
                print(shown or row)
            if len(rows) > 12:
                print(f"... {len(rows) - 12} more rows")
            ok = bool(expect(rows))
            timings.append((label, elapsed, len(rows), ok))
            check(label, ok, f"{elapsed:.2f}s  {len(rows)} row(s)")
        except Exception as exc:
            timings.append((label, None, 0, False))
            check(label, False, f"{type(exc).__name__}: {exc}"[:300])
    return created.full_name


## 5. Count by space and component

Arguments: `space STRING`, `componentName STRING`. The example literals are lab A and `MOISTURE_CONTENT`. The calls use lab B and `PH` as well.


In [ ]:
exercise(
    "Count by space and component",
    "lims_fn_count_by_space_and_component",
    f'''
    SELECT count(*) AS n
    FROM {VIEW}
    WHERE space = '{LAB_A}' AND componentName = 'MOISTURE_CONTENT'
    ''',
    [
        ("lab A, MOISTURE_CONTENT", f"SELECT * FROM {{fn}}(space => '{LAB_A}', componentName => 'MOISTURE_CONTENT')", one_count(6_000)),
        ("lab B, MOISTURE_CONTENT", f"SELECT * FROM {{fn}}(space => '{LAB_B}', componentName => 'MOISTURE_CONTENT')", one_count(4_000)),
        ("lab A, PH", f"SELECT * FROM {{fn}}(space => '{LAB_A}', componentName => 'PH')", one_count(6_000)),
        ("lab B, PH", f"SELECT * FROM {{fn}}(space => '{LAB_B}', componentName => 'PH')", one_count(4_000)),
    ],
    absent=[LAB_A, "MOISTURE_CONTENT"],
)


## 6. Count by a list of spaces

Argument: `space ARRAY<STRING>`. One call passes both labs. The others pass a single lab inside an array.


In [ ]:
exercise(
    "Count by spaces",
    "lims_fn_count_by_spaces",
    f"SELECT count(*) AS n FROM {VIEW} WHERE space IN ('{LAB_A}', '{LAB_B}')",
    [
        ("both labs", f"SELECT * FROM {{fn}}(space => array('{LAB_A}', '{LAB_B}'))", one_count(100_000)),
        ("lab A only", f"SELECT * FROM {{fn}}(space => array('{LAB_A}'))", one_count(60_000)),
        ("lab B only", f"SELECT * FROM {{fn}}(space => array('{LAB_B}'))", one_count(40_000)),
    ],
    absent=[LAB_A, LAB_B],
)


## 7. Count by a list of components

Argument: `componentName ARRAY<STRING>`.


In [ ]:
exercise(
    "Count by components",
    "lims_fn_count_by_components",
    f"SELECT count(*) AS n FROM {VIEW} WHERE componentName IN ('MOISTURE_CONTENT', 'PH')",
    [
        ("moisture and PH", "SELECT * FROM {fn}(componentName => array('MOISTURE_CONTENT', 'PH'))", one_count(20_000)),
        ("moisture only", "SELECT * FROM {fn}(componentName => array('MOISTURE_CONTENT'))", one_count(10_000)),
        ("three components", "SELECT * FROM {fn}(componentName => array('MOISTURE_CONTENT', 'PH', 'DENSITY'))", one_count(30_000)),
    ],
    absent=["MOISTURE_CONTENT", "PH"],
)


## 8. Space plus a component list, and a numeric range

Two functions. The first takes `space STRING` and `componentName ARRAY<STRING>`. The second adds `numericValue_gte DOUBLE` and `numericValue_lte DOUBLE`.


In [ ]:
exercise(
    "Space and component list",
    "lims_fn_count_by_space_and_components",
    f'''
    SELECT count(*) AS n
    FROM {VIEW}
    WHERE space = '{LAB_A}' AND componentName IN ('MOISTURE_CONTENT', 'PH')
    ''',
    [
        ("lab A, moisture and PH", f"SELECT * FROM {{fn}}(space => '{LAB_A}', componentName => array('MOISTURE_CONTENT', 'PH'))", one_count(12_000)),
        ("lab B, moisture only", f"SELECT * FROM {{fn}}(space => '{LAB_B}', componentName => array('MOISTURE_CONTENT'))", one_count(4_000)),
        ("lab B, three components", f"SELECT * FROM {{fn}}(space => '{LAB_B}', componentName => array('MOISTURE_CONTENT', 'PH', 'DENSITY'))", one_count(12_000)),
    ],
    absent=[LAB_A, "MOISTURE_CONTENT", "PH"],
)
exercise(
    "Space, component, and numeric range",
    "lims_fn_count_in_range",
    f'''
    SELECT count(*) AS n
    FROM {VIEW}
    WHERE space = '{LAB_A}'
      AND componentName = 'MOISTURE_CONTENT'
      AND numericValue >= 0
      AND numericValue <= 1000
    ''',
    [
        ("lab A moisture, 0 to 1000", f"SELECT * FROM {{fn}}(space => '{LAB_A}', componentName => 'MOISTURE_CONTENT', numericValue_gte => 0, numericValue_lte => 1000)", one_count(4_200)),
        ("lab B PH, 0 to 1000", f"SELECT * FROM {{fn}}(space => '{LAB_B}', componentName => 'PH', numericValue_gte => 0, numericValue_lte => 1000)", one_count(3_200)),
        ("lab A moisture, 1000 to 2000", f"SELECT * FROM {{fn}}(space => '{LAB_A}', componentName => 'MOISTURE_CONTENT', numericValue_gte => 1000, numericValue_lte => 2000)", one_count(0)),
    ],
    absent=[LAB_A, "MOISTURE_CONTENT"],
)


## 9. A filter that stays in the function body

`aliquotId IS NULL` is part of the query shape, so it is not an argument. `space` is. Lab A has 12,000 rows with a null aliquot. Lab B has 8,000.


In [ ]:
exercise(
    "Space, with null aliquot fixed in the body",
    "lims_fn_count_null_aliquot",
    f"SELECT count(*) AS n FROM {VIEW} WHERE space = '{LAB_A}' AND aliquotId IS NULL",
    [
        ("lab A", f"SELECT * FROM {{fn}}(space => '{LAB_A}')", one_count(12_000)),
        ("lab B", f"SELECT * FROM {{fn}}(space => '{LAB_B}')", one_count(8_000)),
    ],
    absent=[LAB_A],
)


## 10. Lookup by external id

One function takes `external_id STRING`. The other takes `external_id ARRAY<STRING>`. These return rows, not a count. `result_000001` and `result_060001` are both `MOISTURE_CONTENT`, in lab A and lab B.


In [ ]:
exercise(
    "Lookup one external id",
    "lims_fn_by_external_id",
    f"SELECT external_id, componentName FROM {VIEW} WHERE external_id = 'result_000001'",
    [
        ("result_000001", "SELECT space, external_id, componentName FROM {fn}(external_id => 'result_000001')", lambda rows: len(rows) == 1 and rows[0].get("componentName") == "MOISTURE_CONTENT" and rows[0].get("space") == LAB_A),
        ("result_060001", "SELECT space, external_id, componentName FROM {fn}(external_id => 'result_060001')", lambda rows: len(rows) == 1 and rows[0].get("space") == LAB_B and rows[0].get("componentName") == "MOISTURE_CONTENT"),
    ],
    absent=["result_000001"],
)
exercise(
    "Lookup a list of external ids",
    "lims_fn_by_external_ids",
    f"SELECT external_id FROM {VIEW} WHERE external_id IN ('result_000001', 'result_000002')",
    [
        ("two ids", "SELECT external_id FROM {fn}(external_id => array('result_000001', 'result_000002'))", lambda rows: {row.get("external_id") for row in rows} == {"result_000001", "result_000002"}),
        ("one id in an array", "SELECT external_id FROM {fn}(external_id => array('result_060001'))", lambda rows: len(rows) == 1 and rows[0].get("external_id") == "result_060001"),
    ],
    absent=["result_000001", "result_000002"],
)


## 11. A row limit

Arguments: `space STRING`, `row_limit BIGINT`. The same function returns 2 rows or 5 rows depending on the argument.


In [ ]:
exercise(
    "Limited rows in one space",
    "lims_fn_limit_by_space",
    f"SELECT external_id FROM {VIEW} WHERE space = '{LAB_A}' LIMIT 2",
    [
        ("lab A, 2 rows", f"SELECT external_id, space FROM {{fn}}(space => '{LAB_A}', row_limit => 2)", lambda rows: len(rows) == 2 and {row.get("space") for row in rows} == {LAB_A}),
        ("lab B, 5 rows", f"SELECT external_id, space FROM {{fn}}(space => '{LAB_B}', row_limit => 5)", lambda rows: len(rows) == 5 and {row.get("space") for row in rows} == {LAB_B}),
    ],
    absent=[LAB_A],
)


## 12. Min and max

`min` and `max` are separate functions. Each takes `space STRING`. Lab A and lab B both span 1.0 to 1500.0.


In [ ]:
exercise(
    "Minimum numericValue",
    "lims_fn_min_numeric",
    f"SELECT min(numericValue) AS low FROM {VIEW} WHERE space = '{LAB_A}'",
    [
        ("min lab A", f"SELECT * FROM {{fn}}(space => '{LAB_A}')", lambda rows: len(rows) == 1 and float(rows[0]["min_numericValue"]) == 1.0),
        ("min lab B", f"SELECT * FROM {{fn}}(space => '{LAB_B}')", lambda rows: len(rows) == 1 and float(rows[0]["min_numericValue"]) == 1.0),
    ],
    absent=[LAB_A],
)
exercise(
    "Maximum numericValue",
    "lims_fn_max_numeric",
    f"SELECT max(numericValue) AS high FROM {VIEW} WHERE space = '{LAB_A}'",
    [
        ("max lab A", f"SELECT * FROM {{fn}}(space => '{LAB_A}')", lambda rows: len(rows) == 1 and float(rows[0]["max_numericValue"]) == 1500.0),
        ("max lab B", f"SELECT * FROM {{fn}}(space => '{LAB_B}')", lambda rows: len(rows) == 1 and float(rows[0]["max_numericValue"]) == 1500.0),
    ],
    absent=[LAB_A],
)


## 13. GROUP BY

The grouped columns stay in the function body. Only the `WHERE` literals become arguments.

- `lims_fn_count_by_component` groups by `componentName` and takes `space STRING`. Each component is 6,000 in lab A and 4,000 in lab B.
- `lims_fn_count_by_component_in_spaces` takes `space ARRAY<STRING>`.
- `lims_fn_count_by_space_for_component` groups by `space, componentName` and takes `componentName STRING`. One component returns two rows.


In [ ]:
exercise(
    "GROUP BY componentName, filter by space",
    "lims_fn_count_by_component",
    f'''
    SELECT componentName, count(*) AS n
    FROM {VIEW}
    WHERE space = '{LAB_A}'
    GROUP BY componentName
    ''',
    [
        ("groups in lab A", f"SELECT * FROM {{fn}}(space => '{LAB_A}')", grouped(10, "6000")),
        ("groups in lab B", f"SELECT * FROM {{fn}}(space => '{LAB_B}')", grouped(10, "4000")),
    ],
    absent=[LAB_A],
)
exercise(
    "GROUP BY componentName, filter by a list of spaces",
    "lims_fn_count_by_component_in_spaces",
    f'''
    SELECT componentName, count(*) AS n
    FROM {VIEW}
    WHERE space IN ('{LAB_A}', '{LAB_B}')
    GROUP BY componentName
    ''',
    [
        ("groups in both labs", f"SELECT * FROM {{fn}}(space => array('{LAB_A}', '{LAB_B}'))", grouped(10, "10000")),
        ("groups in lab B only", f"SELECT * FROM {{fn}}(space => array('{LAB_B}'))", grouped(10, "4000")),
    ],
    absent=[LAB_A, LAB_B],
)
exercise(
    "GROUP BY space and componentName, filter by component",
    "lims_fn_count_by_space_for_component",
    f'''
    SELECT space, componentName, count(*) AS n
    FROM {VIEW}
    WHERE componentName = 'MOISTURE_CONTENT'
    GROUP BY space, componentName
    ''',
    [
        (
            "moisture in both labs",
            "SELECT * FROM {fn}(componentName => 'MOISTURE_CONTENT')",
            lambda rows: len(rows) == 2 and { (row.get("space"), str(row.get("count_externalId"))) for row in rows } == {(LAB_A, "6000"), (LAB_B, "4000")},
        ),
        (
            "PH in both labs",
            "SELECT * FROM {fn}(componentName => 'PH')",
            lambda rows: len(rows) == 2 and { (row.get("space"), str(row.get("count_externalId"))) for row in rows } == {(LAB_A, "6000"), (LAB_B, "4000")},
        ),
    ],
    absent=["MOISTURE_CONTENT"],
)


## 14. A query that must not create a function

`componentName` is selected but the query groups by `space`. `create_sql_function` raises, and Unity Catalog does not get a function of that name.


In [ ]:
bad_name = "lims_fn_bad_group"
try:
    generator.create_sql_function(
        bad_name,
        f"SELECT componentName, count(*) AS n FROM {VIEW} GROUP BY space",
        secret_scope=SECRET_SCOPE,
    )
    check("Mismatched GROUP BY is rejected", False, "function was created")
except QueryNotPushdownCompatible as exc:
    check("Mismatched GROUP BY is rejected", True, str(exc)[:180])

try:
    workspace_client.functions.get(f"{CATALOG}.{SCHEMA}.{bad_name}")
    check("Rejected name is absent from Unity Catalog", False)
except NotFound:
    check("Rejected name is absent from Unity Catalog", True)


## 15. Latency


In [ ]:
if not timings:
    print("No function calls were timed.")
else:
    summary = spark.createDataFrame(
        [
            (name, None if seconds is None else round(seconds, 2), row_count, "PASS" if ok else "FAIL")
            for name, seconds, row_count, ok in timings
        ],
        "call string, seconds double, rows long, result string",
    )
    summary.show(60, truncate=False)
    display(summary)
failed = [name for name, ok, _detail in results if not ok]
print(f"{len(results) - len(failed)} passed, {len(failed)} failed")
if failed:
    print("failed:", ", ".join(failed))


## 16. SQL examples

Run these after the functions above exist. Each cell is a SQL call, with the arguments written out.


Count one component in one lab. Expect 6000.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_by_space_and_component(
  space => 'inst_lims_result_lab_a',
  componentName => 'MOISTURE_CONTENT'
)


Same function, other arguments. Lab B PH is 4000.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_by_space_and_component(
  space => 'inst_lims_result_lab_b',
  componentName => 'PH'
)


Pass both labs as an array. Expect 100000.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_by_spaces(
  space => array('inst_lims_result_lab_a', 'inst_lims_result_lab_b')
)


Two components across both labs. Expect 20000.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_by_components(
  componentName => array('MOISTURE_CONTENT', 'PH')
)


Numeric range. Lab B PH between 0 and 1000 is 3200.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_in_range(
  space => 'inst_lims_result_lab_b',
  componentName => 'PH',
  numericValue_gte => 0,
  numericValue_lte => 1000
)


Null aliquots stay in the function body. Only space is an argument. Lab A is 12000.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_null_aliquot(
  space => 'inst_lims_result_lab_a'
)


Look up one external id.


In [ ]:
%sql
SELECT space, external_id, componentName
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_by_external_id(
  external_id => 'result_000001'
)


Ask for five rows from lab B.


In [ ]:
%sql
SELECT space, external_id
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_limit_by_space(
  space => 'inst_lims_result_lab_b',
  row_limit => 5
)


Minimum numeric value in lab A. Expect 1.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_min_numeric(
  space => 'inst_lims_result_lab_a'
)


Ten component groups in lab A, 6000 each.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_by_component(
  space => 'inst_lims_result_lab_a'
)


Moisture in both labs: 6000 in lab A and 4000 in lab B.


In [ ]:
%sql
SELECT *
FROM f0connectortest.dm_dom_lims_result_limsresult_dom_v1.lims_fn_count_by_space_for_component(
  componentName => 'MOISTURE_CONTENT'
)
